# RISE HCP-D neural network fine-tuning

Fine-tune repeated HCP pretrained model weights on HCP-D, then optionally evaluate and export HCP-D embeddings.

In [ ]:
from pathlib import Path
import socket
import subprocess
import sys
import torch

PROJECT_ROOT = Path('..').resolve()
from rise.neural_network.finetune_config import parse_finetune_config

print('Host:', socket.gethostname())
print('PyTorch:', torch.__version__)
print('CUDA build:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
if torch.cuda.is_available():
    for index in range(torch.cuda.device_count()):
        print(f'GPU {index}:', torch.cuda.get_device_name(index))
else:
    print('No GPU is visible; neural-network steps will run on CPU.')

In [ ]:
FINETUNE_MODULE = 'rise.neural_network.finetune'
TEST_MODULE = 'rise.neural_network.test'
TRANSFORM_MODULE = 'rise.neural_network.transform'
CONFIG_FILE = PROJECT_ROOT / 'configs/hcpd_network_finetune_parameters.jsonc'

# All paths, source/target models, and optimization parameters come from CONFIG_FILE.
nn_config = parse_finetune_config(['--config', str(CONFIG_FILE)])
configured_device = nn_config.device
if configured_device.lower() == 'auto':
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
elif configured_device.startswith('cuda') and not torch.cuda.is_available():
    print(
        f'Configured device {configured_device!r} is unavailable; '
        'falling back to CPU.'
    )
    DEVICE = 'cpu'
else:
    DEVICE = configured_device

TRAIN_DATA = nn_config.train_data
TEST_DATA = nn_config.test_data
TEST_GROUP_DATA = nn_config.test_group_data

print('Configuration:', CONFIG_FILE)
print('Pretrained HCP model:', nn_config.pretrained_model_name)
print('Fine-tuned HCP-D model:', nn_config.model_name)
print('Epochs:', nn_config.epochs)
print('Learning rate:', nn_config.learning_rate)
print('Validation ratio:', nn_config.validation_ratio)
print('Configured device:', configured_device)
print('Runtime device:', DEVICE)

for module in (FINETUNE_MODULE, TEST_MODULE, TRANSFORM_MODULE):
    subprocess.run([sys.executable, '-m', module, '--config', str(CONFIG_FILE), '--help'], check=True)

### Finetune

In [ ]:
if not TRAIN_DATA.is_file():
    raise FileNotFoundError(f'Missing HCP-D training input: {TRAIN_DATA}')
subprocess.run([
    sys.executable, '-m', FINETUNE_MODULE,
    '--config', str(CONFIG_FILE),
    '--device', DEVICE,
], check=True)


## Model evaluation

In [ ]:
if not TEST_GROUP_DATA.is_file():
    raise FileNotFoundError(f'Missing HCP-D group test input: {TEST_GROUP_DATA}')
subprocess.run([
    sys.executable, '-m', TEST_MODULE,
    '--config', str(CONFIG_FILE),
    '--device', DEVICE,
], check=True)


## Extract parcel embeddings

In [ ]:
TRANSFORM_SPLIT = 'both'
for data_path in (TRAIN_DATA, TEST_DATA):
    if not data_path.is_file():
        raise FileNotFoundError(f'Missing HCP-D transform input: {data_path}')
subprocess.run([
    sys.executable, '-m', TRANSFORM_MODULE,
    '--config', str(CONFIG_FILE),
    '--device', DEVICE,
    '--split', TRANSFORM_SPLIT,
], check=True)
